# Data Preparation for Machine Learning

This notebook provides a step-by-step guide to cleaning tree data from Grenoble in order to predict the planting year.

## Objective
Prepare the data to train a Machine Learning model that can predict missing tree planting years.

## Notebook Outline
1. Import libraries and load data  
2. Initial data exploration  
3. Data cleaning  
4. Save the data as a pickle file

## 1. Importing Libraries and Loading Data

We use several essential Python libraries for data analysis:
- pandas: for data manipulation  
- numpy: for numerical computations  
- matplotlib and seaborn: for visualization  
- sklearn: for preparing data for Machine Learning

In [ ]:
#%pip install -r '../requirements.txt'
%conda init
%conda activate sklearn-env

# Libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Loading the Data
df = pd.read_csv('../arbres_grenoble_epsg4326.csv', sep=',', encoding='utf-8')

# Display the first few rows
print('Data preview:')
display(df.head())

# Dataset information
print('\nDataset information:')
display(df.info())

In [ ]:
df_traitch = df['traitementchenilles'].dropna()
df_traitch.value_counts()

In [ ]:
df['raisondeplantation'].value_counts()

In [ ]:
for i in df.columns:
    print(df[i].value_counts())

## 2. Initial Data Exploration

Before starting the cleaning process, it's crucial to understand our data:
- What columns are available?  
- What types of data do we have?  
- Are there any missing values?

In [ ]:
# Descriptive Statistics
df.dtypes
# Analysis of Missing Values
x=df[df['anneedeplantation']==2000].isna().sum()/len(df[df['anneedeplantation']==2000])*100
print(x)

In [ ]:
df[df['anneedeplantation']==1900]
df['anneedeplantation'].value_counts().index.sort_values()

### Visualisation of missing values

In [ ]:
# Creation of a heatmap to visualize missing values
#sns.heatmap(df.isna())
percent_missing=pd.DataFrame()
for i in df['anneedeplantation'].value_counts().index.sort_values():
    percent_missing[i]=df[df['anneedeplantation']==int(i)].isna().sum()/len(df[df['anneedeplantation']==int(i)])*100

percent_missing

sns.heatmap(percent_missing, cmap='Greens')

## 3. Data Cleaning

### 3.1 After the initial analysis, we will:
1. Remove rows without a planting year **'annedeplantation'**
2. Drop columns that are irrelevant for prediction  
3. Extract geographic coordinates

In [ ]:
# 1. Remove rows without a planting year **'annedeplantation'**

""" df = df[df['anneedeplantation'].notna()]

print(df['anneedeplantation'].isna().sum())

x=df.isna().sum()/len(df)*100
x = x.reset_index()
x = x[x[0] == 100]
todrop=list(x['index']) 

df = df.drop(columns=todrop)

todrop2 = ['genre','genre_desc','elem_point_id','code','nom','categorie','categorie_desc','bien_reference','collectivite','typenature']
df = df.drop(columns=todrop2)

 """
 
# remove also columns with only one unique value, all unique values, or no values, but be careful with important columns like 'geo_point_2d'

location=df["geo_point_2d"].copy()

todrop3 = df.loc[:, df.nunique().isin([1, len(df), 0])].columns

df = df.drop(columns=todrop3)
df["location"]=location



### 🔍 3.2 Cleaning Unnecessary Columns

We will:

- Remove columns with more than **40% missing values**  
- Remove columns that are **too specific** (e.g., unique identifiers), which don't provide useful information for machine learning  
- Remove columns that contain redundant or unhelpful information

In [ ]:
# 2. Extract geographic coordinates

print(df.columns)
#%conda install geopy
import json
from geopy.distance import geodesic

print(df["location"].head())

#df["location"]= df['location'].apply(lambda x: (json.loads(x)['coordinates']))
df["location"] = df["location"].apply(lambda x: x.split(","))
print(df["location"].head())
df["latitude"] = df["location"].apply(lambda x: x[0])
print(df["latitude"].head())
df["longitude"] = df["location"].apply(lambda x: x[1])
print(df["longitude"].head())
print(df["location"])


mask = (
    df["latitude"].str.strip().eq("") |
    df["longitude"].str.strip().eq("") |
    df["latitude"].isnull() |
    df["longitude"].isnull()
)
df.loc[mask, ["location", "latitude", "longitude"]]

In [ ]:

# transform latitude and longitude to float
df["latitude"] = pd.to_numeric(df["latitude"].str.strip(), errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"].str.strip(), errors="coerce")


print(df["latitude"].head())
print(df["longitude"].head())

In [ ]:
# 3. Remove overly specific columns, such as the 'elem_point_id' column

display(df.info())

df = df.drop('location', axis=1).copy()

display(df.info())


In [ ]:
for col in df.columns:
     print(f"\n{col}")
     print(df[col].value_counts())
     print(df[col].dtype)

In [ ]:
df['adr_secteur'] = df['adr_secteur'].astype(str)
df['identifiantplu'] = df['identifiantplu'].astype(str)

# drop les colonnes qui sont les memes que d'autres colonnes

df.drop(columns=['code_parent', 'sous_categorie'], inplace=True)    

df.dtypes

In [ ]:
#je veux recoder les missing values en 'Unknown'
cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns
df[cat_cols] = df[cat_cols].astype("str").fillna("Unknown")

# Vérification après le recodage des missing values
df.isnull().sum()

# je veux aussi recoder les categories dans les str variables avec moins de 30 representant  en "Rare"
for col in df.select_dtypes(include='str').columns:
    value_counts = df[col].value_counts()
    rare_values = value_counts[value_counts <= 30].index
    df[col] = df[col].replace(rare_values, 'Rare')
    
# Vérification finale des types de données
df.dtypes

for col in df.columns:
     print(f"\n{col}")
     print(df[col].value_counts())
     print(df[col].dtype)
     print(df[col].nunique())
    

## 4. Save to data to a pickle file

In [ ]:
# Save to data to a pickle file

df.to_pickle('arbres_clean_Alexia.pkl')

## Conclusion

We have completed several important cleaning steps:
1. Removed irrelevant columns or those with too many missing values  
2. Extracted geographic coordinates  

The data is now ready for the next steps of the project.